In [2]:
import os
import torch
import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

os.makedirs('./data', exist_ok=True)
os.makedirs('./results', exist_ok=True)

Using device: cpu


In [2]:
import zipfile

DATA_DIR = './data'
ZIP_PATH = './data/ships-aerial-images.zip'
DATASET_NAME = 'ships-aerial-images'
dataset_root = os.path.join(DATA_DIR, DATASET_NAME)

if not os.path.exists(dataset_root):
    if not os.path.exists(ZIP_PATH):
        print("Dataset not found.")
        print("1. Download ships-aerial-images.zip from Kaggle")
        print("2. Place it in ./data/")
        print("Link: https://www.kaggle.com/datasets/siddharthkumarsah/ships-aerial-images")
    else:
        print("Unpacking dataset...")
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
        print("Unpacking complete.")
else:
    print("Dataset already extracted.")

print(f"\nDataset root: {dataset_root}")
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset_root, split, 'images')
    lbl_dir = os.path.join(dataset_root, split, 'labels')
    n_img = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_lbl = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    print(f"{split}: {n_img} images, {n_lbl} labels")

Dataset already extracted.

Dataset root: ./data\ships-aerial-images
train: 9697 images, 9697 labels
valid: 2165 images, 2165 labels
test: 1573 images, 1573 labels


In [6]:
yaml_path = os.path.join(dataset_root, 'data.yaml')

yaml_content = f"""path: {dataset_root}
train: train/images
val: valid/images
test: test/images

nc: 1
names:
  0: ship
"""

with open(yaml_path, 'w') as f:
    f.write(yaml_content.strip())

print(f"data.yaml created: {yaml_path}")

data.yaml created: ./data\ships-aerial-images\data.yaml


1. Выбор начальных условий

Датасет: Ships in Aerial Images (Kaggle), 1 класс ship, разметка в YOLO-формате.
Обоснование: задача имеет прикладной смысл для морского мониторинга, навигационной безопасности и контроля акваторий.
Метрики: mAP50 и mAP50-95 как базовые метрики детекции; Precision и Recall для контроля баланса ложных срабатываний и пропусков; F1 как итоговый компромисс между Precision и Recall.

In [7]:
print("Starting Baseline Training (YOLOv11n)...")

model_baseline = YOLO('yolo11n.pt')

results_baseline = model_baseline.train(
    data=yaml_path,
    epochs=10,
    imgsz=64,
    batch=16,
    device=DEVICE,
    name='ships_baseline',
    project='./results',
    patience=3,
    save=True,
    save_period=1,
    cache=False,
    workers=4,
    verbose=False,
    seed=42,
    exist_ok=True
)

print("Baseline training finished.")

Starting Baseline Training (YOLOv11n)...
Ultralytics 8.4.41  Python-3.11.4 torch-2.11.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./data\ships-aerial-images\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=64, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ships_baseline, nbs=64, nms=False, opset=None, optimiz

In [10]:
print("Evaluating Baseline Model...")

baseline_model_path = './runs/detect/results/ships_baseline/weights/best.pt'
baseline_csv_path = './runs/detect/results/ships_baseline/results.csv'

model_bl = YOLO(baseline_model_path)

metrics_bl = model_bl.val(data=yaml_path, split='val', verbose=False)

baseline_metrics = {
    'mAP50': float(metrics_bl.box.map50.item()),
    'mAP50-95': float(metrics_bl.box.map.item()),
    'Precision': float(metrics_bl.box.mp.item()),
    'Recall': float(metrics_bl.box.mr.item()),
    'F1-Score': float(metrics_bl.box.f1.item())
}

print("\n" + "="*50)
print("BASELINE RESULTS")
print("="*50)
for k, v in baseline_metrics.items():
    print(f"{k}: {v:.4f}")
print("="*50)

if os.path.exists(baseline_csv_path):
    df_bl = pd.read_csv(baseline_csv_path)
    if len(df_bl) > 1:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        axes[0, 0].plot(df_bl['epoch'], df_bl['metrics/mAP50(B)'], 'b-', linewidth=2)
        axes[0, 0].set_title('mAP@0.5')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].grid(True, alpha=0.3)
        
        axes[0, 1].plot(df_bl['epoch'], df_bl['metrics/precision(B)'], 'g-', linewidth=2)
        axes[0, 1].set_title('Precision')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].grid(True, alpha=0.3)
        
        axes[1, 0].plot(df_bl['epoch'], df_bl['metrics/recall(B)'], 'r-', linewidth=2)
        axes[1, 0].set_title('Recall')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].grid(True, alpha=0.3)
        
        axes[1, 1].plot(df_bl['epoch'], df_bl['train/box_loss'], 'orange', linewidth=2)
        axes[1, 1].set_title('Box Loss')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('./results/baseline_metrics.png', dpi=300, bbox_inches='tight')
        plt.show()
    else:
        print("Warning: results.csv has only 1 row. Plots skipped.")

Evaluating Baseline Model...
Ultralytics 8.4.41  Python-3.11.4 torch-2.11.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 193.9206.7 MB/s, size: 25.0 KB)
val: Scanning C:\Users\Пользователь\MAI_study\cyberphys\data\ships-aerial-images\valid\labels.cache... 2165 images, 68 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2165/2165  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 172, len(boxes) = 3720. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 136/136 7.0it/s 19.5s0.1s
                   all       2165       3720      0.474      0.216      0.197      0.093
Speed: 0.1ms preprocess, 4.5ms inference, 0.0ms loss

<Figure size 1400x1000 with 4 Axes>

2a-2b. Создание бейзлайна и оценка качества

Бейзлайн построен на YOLOv11n. Модель обучена на train и оценена на val по метрикам mAP50, mAP50-95, Precision, Recall, F1. Эти результаты используются как точка сравнения для последующих улучшений.

In [12]:
print("Starting Improved Baseline Training (YOLOv11s + Optimized Params)...")

model_improved = YOLO('yolo11s.pt')

results_improved = model_improved.train(
    data=yaml_path,
    epochs=15,
    imgsz=96,
    batch=16,
    device=DEVICE,
    name='ships_improved',
    project='./results',
    patience=4,
    save=True,
    save_period=1,
    cache='ram',
    workers=4,
    verbose=False,
    seed=42,
    exist_ok=True,
    optimizer='AdamW',
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    cos_lr=True,
    close_mosaic=5,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    flipud=0.5,
    fliplr=0.5,
    mosaic=0.8
)

print("Improved training finished.")

Starting Improved Baseline Training (YOLOv11s + Optimized Params)...
Ultralytics 8.4.41  Python-3.11.4 torch-2.11.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=./data\ships-aerial-images\data.yaml, degrees=15.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=96, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=0.8, multi_scale=0.0, name=ships_improved, nbs=64, nms=Fa

In [13]:
print("Evaluating Improved Model...")

improved_model_path = './runs/detect/results/ships_improved/weights/best.pt'
improved_csv_path = './runs/detect/results/ships_improved/results.csv'

model_imp = YOLO(improved_model_path)

metrics_imp = model_imp.val(data=yaml_path, split='val', verbose=False)

improved_metrics = {
    'mAP50': float(metrics_imp.box.map50.item()),
    'mAP50-95': float(metrics_imp.box.map.item()),
    'Precision': float(metrics_imp.box.mp.item()),
    'Recall': float(metrics_imp.box.mr.item()),
    'F1-Score': float(metrics_imp.box.f1.item())
}

print("\n" + "="*50)
print("IMPROVED BASELINE RESULTS")
print("="*50)
for k, v in improved_metrics.items():
    print(f"{k}: {v:.4f}")
print("="*50)

if os.path.exists(baseline_csv_path) and os.path.exists(improved_csv_path):
    df_bl = pd.read_csv(baseline_csv_path)
    df_imp = pd.read_csv(improved_csv_path)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    axes[0, 0].plot(df_bl['epoch'], df_bl['metrics/mAP50(B)'], 'b-', linewidth=2, label='Baseline')
    axes[0, 0].plot(df_imp['epoch'], df_imp['metrics/mAP50(B)'], 'b--', linewidth=2, label='Improved')
    axes[0, 0].set_title('mAP@0.5')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()
    
    axes[0, 1].plot(df_bl['epoch'], df_bl['metrics/precision(B)'], 'g-', linewidth=2, label='Baseline')
    axes[0, 1].plot(df_imp['epoch'], df_imp['metrics/precision(B)'], 'g--', linewidth=2, label='Improved')
    axes[0, 1].set_title('Precision')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()
    
    axes[1, 0].plot(df_bl['epoch'], df_bl['metrics/recall(B)'], 'r-', linewidth=2, label='Baseline')
    axes[1, 0].plot(df_imp['epoch'], df_imp['metrics/recall(B)'], 'r--', linewidth=2, label='Improved')
    axes[1, 0].set_title('Recall')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()
    
    axes[1, 1].plot(df_bl['epoch'], df_bl['train/box_loss'], 'orange', linewidth=2, label='Baseline')
    axes[1, 1].plot(df_imp['epoch'], df_imp['train/box_loss'], 'orange', linestyle='--', linewidth=2, label='Improved')
    axes[1, 1].set_title('Box Loss')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.savefig('./results/comparison_metrics.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Warning: CSV files not found. Comparison plots skipped.")

Evaluating Improved Model...
Ultralytics 8.4.41  Python-3.11.4 torch-2.11.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access  (ping: 0.30.0 ms, read: 2.42.0 MB/s, size: 12.7 KB)
val: Scanning C:\Users\Пользователь\MAI_study\cyberphys\data\ships-aerial-images\valid\labels.cache... 2165 images, 68 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2165/2165  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 172, len(boxes) = 3720. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 136/136 3.0it/s 45.8s0.5ss
                   all       2165       3720      0.466      0.258       0.24      0.108
Speed: 0.1ms preprocess, 11.2ms inference, 0.0ms loss,

<Figure size 1400x1000 with 4 Axes>

3a-3g. Улучшение бейзлайна

Гипотезы: более ёмкая модель YOLOv11s, увеличение разрешения, усиление аугментаций и настройка оптимизации. По результатам эксперимента сформирован improved baseline, проведено сравнение с baseline по выбранным метрикам и сделаны выводы о приросте качества.

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import box_iou, batched_nms
import cv2
import numpy as np
import random
from tqdm import tqdm
import matplotlib.pyplot as plt

NUM_CLASSES = 1
IMG_SIZE = 64
BATCH_SIZE = 16
EPOCHS = 10
LR = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CUSTOM_WEIGHTS_BEST = './results/light_detector.pt'
CUSTOM_WEIGHTS_LAST = './results/light_detector_last.pt'

IMPROVED_IMG_SIZE = 96
IMPROVED_EPOCHS = 15
IMPROVED_BATCH = 16
IMPROVED_LR_MAX = 3e-4
IMPROVED_WD = 2e-4
IMPROVED_WARMUP_EPOCHS = 3
IMPROVED_CLOSE_AUG_EPOCHS = 5
CUSTOM_WEIGHTS_IMPROVED_BEST = './results/light_detector_improved.pt'
CUSTOM_WEIGHTS_IMPROVED_LAST = './results/light_detector_improved_last.pt'

print(f"Using device: {DEVICE}")

Using device: cpu


In [3]:
class LightDataset(Dataset):
    """augment_mode='basic' — только fliplr (как простой бейзлайн). 'improved' — HSV, fliplr/flipud, translate (аналог ships_improved)."""

    def __init__(self, split, img_size=IMG_SIZE, augment=False, augment_mode='basic'):
        self.img_dir = os.path.join(dataset_root, split, 'images')
        self.lbl_dir = os.path.join(dataset_root, split, 'labels')
        self.img_size = img_size
        self.augment = augment
        self.augment_mode = augment_mode
        self.weak_augment = False
        self.files = [f for f in os.listdir(self.img_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.files)

    def _read_yolo_boxes_norm(self, lbl_path):
        entries = []
        if not os.path.exists(lbl_path):
            return entries
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls = int(parts[0])
                    xc, yc, bw, bh = map(float, parts[1:5])
                    xc = float(min(max(xc, 0.0), 1.0))
                    yc = float(min(max(yc, 0.0), 1.0))
                    bw = float(min(max(bw, 1e-6), 1.0))
                    bh = float(min(max(bh, 1e-6), 1.0))
                    entries.append((cls, xc, yc, bw, bh))
        return entries

    def _to_pixel_boxes(self, entries):
        boxes, labels = [], []
        for cls, xc, yc, bw, bh in entries:
            x1 = (xc - bw / 2) * self.img_size
            y1 = (yc - bh / 2) * self.img_size
            x2 = (xc + bw / 2) * self.img_size
            y2 = (yc + bh / 2) * self.img_size
            x1, y1 = max(0.0, x1), max(0.0, y1)
            x2, y2 = min(float(self.img_size), x2), min(float(self.img_size), y2)
            if x2 <= x1 + 1.0 or y2 <= y1 + 1.0:
                continue
            boxes.append([x1, y1, x2, y2])
            labels.append(cls)
        if not boxes:
            return torch.zeros((0, 4)), torch.zeros(0, dtype=torch.long)
        return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.long)

    def _apply_improved_hsv(self, img_rgb):
        hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
        rh, rs, rv = np.random.uniform(-1, 1, size=3)
        hsv[:, :, 0] = (hsv[:, :, 0] + rh * 0.015 * 180.0) % 180.0
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * (1.0 + rs * 0.35), 0.0, 255.0)
        hsv[:, :, 2] = np.clip(hsv[:, :, 2] * (1.0 + rv * 0.20), 0.0, 255.0)
        return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img_path = os.path.join(self.img_dir, fname)
        lbl_path = os.path.join(self.lbl_dir, fname.replace('.jpg', '.txt'))

        img = cv2.imread(img_path)
        if img is None:
            img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))

        entries = list(self._read_yolo_boxes_norm(lbl_path))

        if self.augment:
            if self.augment_mode == 'basic':
                if random.random() < 0.5:
                    img = cv2.flip(img, 1)
                    entries = [(c, 1.0 - xc, yc, bw, bh) for c, xc, yc, bw, bh in entries]
            elif self.augment_mode == 'improved':
                img = self._apply_improved_hsv(img)
                if not self.weak_augment:
                    if random.random() < 0.5:
                        img = cv2.flip(img, 1)
                        entries = [(c, 1.0 - xc, yc, bw, bh) for c, xc, yc, bw, bh in entries]
                    if random.random() < 0.5:
                        img = cv2.flip(img, 0)
                        entries = [(c, xc, 1.0 - yc, bw, bh) for c, xc, yc, bw, bh in entries]

                    tx, ty = np.random.uniform(-0.1, 0.1, size=2)
                    tx_px = int(round(tx * self.img_size))
                    ty_px = int(round(ty * self.img_size))
                    M = np.float32([[1, 0, tx_px], [0, 1, ty_px]])
                    img = cv2.warpAffine(
                        img,
                        M,
                        (self.img_size, self.img_size),
                        flags=cv2.INTER_LINEAR,
                        borderMode=cv2.BORDER_CONSTANT,
                        borderValue=(114, 114, 114),
                    )
                    entries = [
                        (c, float(min(max(xc + tx, 0.0), 1.0)), float(min(max(yc + ty, 0.0), 1.0)), bw, bh)
                        for c, xc, yc, bw, bh in entries
                    ]

        img = torch.from_numpy(img.astype(np.float32) / 255.0).permute(2, 0, 1)
        boxes, labels = self._to_pixel_boxes(entries)
        return img, boxes, labels


def collate_fn(batch):
    images, boxes, labels = zip(*batch)
    return torch.stack(images), list(boxes), list(labels)

In [4]:
class LightDetector(nn.Module):
    def __init__(self, num_classes=1, img_size=IMG_SIZE):
        super().__init__()
        self.num_classes = num_classes
        self.img_size = img_size
        self.num_anchors = 3

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2)
        )

        self.cls_head = nn.Conv2d(128, self.num_anchors * (num_classes + 1), 3, padding=1)
        self.reg_head = nn.Conv2d(128, self.num_anchors * 4, 3, padding=1)

        self._generate_anchors()
        nn.init.normal_(self.reg_head.weight, std=0.001)
        nn.init.zeros_(self.reg_head.bias)

    def _generate_anchors(self):
        stride = 16
        grid = self.img_size // stride
        base_anchor_sizes = [(12, 12), (20, 16), (28, 24)]
        scale = self.img_size / 64.0
        anchor_sizes = [(max(4, int(round(w * scale))), max(4, int(round(h * scale)))) for w, h in base_anchor_sizes]
        anchors = []
        for aw, ah in anchor_sizes:
            for gy in range(grid):
                for gx in range(grid):
                    cx = (gx + 0.5) * stride
                    cy = (gy + 0.5) * stride
                    anchors.append([cx - aw/2, cy - ah/2, cx + aw/2, cy + ah/2])
        self.register_buffer('anchors', torch.tensor(anchors, dtype=torch.float32))

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        cls_out = self.cls_head(x)
        reg_out = self.reg_head(x)
        B, _, H, W = cls_out.shape
        cls_out = cls_out.reshape(B, self.num_anchors, self.num_classes+1, H, W)
        cls_out = cls_out.permute(0,1,3,4,2).reshape(B, -1, self.num_classes+1)
        reg_out = reg_out.reshape(B, self.num_anchors, 4, H, W)
        reg_out = reg_out.permute(0,1,3,4,2).reshape(B, -1, 4)
        return cls_out, reg_out

    def decode(self, cls_pred, reg_pred, conf_thresh=0.3, nms_thresh=0.5):
        B = cls_pred.shape[0]
        results = []
        for b in range(B):
            scores = cls_pred[b].softmax(dim=-1)[:, 1:]
            max_scores, class_ids = scores.max(dim=-1)
            mask = max_scores > conf_thresh
            if not mask.any():
                results.append((torch.zeros(0,4), torch.zeros(0), torch.zeros(0, dtype=torch.long)))
                continue
            anchors = self.anchors[mask]
            deltas = reg_pred[b][mask]
            scores_f = max_scores[mask]
            classes_f = class_ids[mask]

            aw = anchors[:,2] - anchors[:,0]
            ah = anchors[:,3] - anchors[:,1]
            acx = (anchors[:,0] + anchors[:,2]) / 2
            acy = (anchors[:,1] + anchors[:,3]) / 2

            pcx = acx + deltas[:,0] * aw
            pcy = acy + deltas[:,1] * ah
            pw = aw * torch.exp(deltas[:,2].clamp(max=4.0))
            ph = ah * torch.exp(deltas[:,3].clamp(max=4.0))

            boxes = torch.stack([pcx - pw/2, pcy - ph/2, pcx + pw/2, pcy + ph/2], dim=-1)
            boxes = boxes.clamp(0, self.img_size)

            keep = batched_nms(boxes, scores_f, classes_f, nms_thresh)
            results.append((boxes[keep], scores_f[keep], classes_f[keep]))
        return results

In [5]:
def compute_loss(cls_preds, reg_preds, anchors, gt_boxes_list, gt_labels_list, neg_pos_ratio=3):
    device = cls_preds.device
    total_cls = torch.tensor(0.0, device=device)
    total_reg = torch.tensor(0.0, device=device)
    num_pos_total = 0

    for b in range(len(gt_boxes_list)):
        gt_boxes = gt_boxes_list[b].to(device)
        gt_labels = gt_labels_list[b].to(device)

        if len(gt_boxes) == 0:
            targets = torch.zeros(anchors.shape[0], dtype=torch.long, device=device)
            loss_cls = F.cross_entropy(cls_preds[b], targets, reduction='mean')
            total_cls += loss_cls
            continue

        ious = box_iou(anchors, gt_boxes)
        best_iou, best_idx = ious.max(dim=1)
        targets = torch.zeros(anchors.shape[0], dtype=torch.long, device=device)
        pos_mask = best_iou >= 0.5
        targets[pos_mask] = gt_labels[best_idx[pos_mask]] + 1

        best_anchor_per_gt = ious.argmax(dim=0)
        for i in range(len(gt_boxes)):
            a = best_anchor_per_gt[i]
            targets[a] = gt_labels[i] + 1
            pos_mask[a] = True

        num_pos = pos_mask.sum().item()
        if num_pos == 0:
            loss_cls = F.cross_entropy(cls_preds[b], targets, reduction='mean')
            total_cls += loss_cls
            continue

        num_pos_total += num_pos

        cls_loss_all = F.cross_entropy(cls_preds[b], targets, reduction='none')
        neg_loss = cls_loss_all.clone()
        neg_loss[pos_mask] = 0
        _, neg_sorted = neg_loss.sort(descending=True)
        num_neg = min(neg_pos_ratio * num_pos, (~pos_mask).sum().item())
        selected = pos_mask.clone()
        if num_neg > 0:
            selected[neg_sorted[:num_neg]] = True
        total_cls += cls_loss_all[selected].sum()

        matched_gt = gt_boxes[best_idx[pos_mask]]
        pa = anchors[pos_mask]
        aw = pa[:,2] - pa[:,0]
        ah = pa[:,3] - pa[:,1]
        acx = (pa[:,0] + pa[:,2]) / 2
        acy = (pa[:,1] + pa[:,3]) / 2

        gcx = (matched_gt[:,0] + matched_gt[:,2]) / 2
        gcy = (matched_gt[:,1] + matched_gt[:,3]) / 2
        gw = matched_gt[:,2] - matched_gt[:,0]
        gh = matched_gt[:,3] - matched_gt[:,1]

        target_deltas = torch.stack([
            (gcx - acx) / (aw + 1e-6),
            (gcy - acy) / (ah + 1e-6),
            torch.log((gw + 1e-6) / (aw + 1e-6)),
            torch.log((gh + 1e-6) / (ah + 1e-6))
        ], dim=-1)
        target_deltas = torch.nan_to_num(target_deltas, nan=0.0, posinf=4.0, neginf=-4.0).clamp(-4.0, 4.0)

        total_reg += F.smooth_l1_loss(reg_preds[b][pos_mask], target_deltas, reduction='sum')

    if num_pos_total == 0:
        return total_cls / max(len(gt_boxes_list), 1)
    loss = total_cls / max(num_pos_total, 1) + total_reg / max(num_pos_total, 1)
    return torch.nan_to_num(loss, nan=0.0, posinf=1e4, neginf=0.0)

In [6]:
def evaluate_full(model, dataloader, conf_thresh=0.25, nms_thresh=0.5):
    """Оценка ближе к COCO: (1) conf_thresh не занижаем до 0.1 — иначе много FP и «растёт только recall»;
    (2) жадное сопоставление pred→GT: каждый GT считается найденным не более одного раза на порог IoU."""
    model.eval()
    iou_thresholds = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]

    events = []
    total_gt = 0
    img_id = 0

    with torch.no_grad():
        for images, gt_boxes_list, _ in tqdm(dataloader, desc='Evaluating'):
            images = images.to(DEVICE)
            cls_preds, reg_preds = model(images)
            results = model.decode(cls_preds, reg_preds, conf_thresh=conf_thresh, nms_thresh=nms_thresh)

            for b in range(len(images)):
                gt = gt_boxes_list[b]
                if isinstance(gt, torch.Tensor) and gt.numel() > 0:
                    gt_cpu = gt.float().cpu()
                else:
                    gt_cpu = torch.zeros((0, 4), dtype=torch.float32)

                n_gt = int(gt_cpu.shape[0])
                total_gt += n_gt

                pred_boxes, pred_scores, _ = results[b]
                pred_boxes = pred_boxes.cpu()
                pred_scores = pred_scores.cpu()

                for i in range(pred_boxes.shape[0]):
                    events.append((float(pred_scores[i].item()), img_id, pred_boxes[i], gt_cpu))
                img_id += 1

    if total_gt == 0:
        return {'mAP50': 0.0, 'mAP50-95': 0.0, 'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0}

    def _greedy_flags(iou_thresh):
        used_per_image = {}
        order = sorted(range(len(events)), key=lambda k: -events[k][0])
        flags = []
        for k in order:
            score, im, pb, gtb = events[k]
            n_g = gtb.shape[0]
            if n_g == 0:
                flags.append((score, False))
                continue
            if im not in used_per_image:
                used_per_image[im] = torch.zeros(n_g, dtype=torch.bool)
            used = used_per_image[im]
            ious = box_iou(pb.unsqueeze(0), gtb)[0]
            if bool(used.all().item()):
                flags.append((score, False))
                continue
            ious_masked = ious.masked_fill(used, -1.0)
            j = int(ious_masked.argmax().item())
            if float(ious_masked[j].item()) >= iou_thresh:
                used[j] = True
                flags.append((score, True))
            else:
                flags.append((score, False))
        flags.sort(key=lambda x: -x[0])
        return [m for _, m in flags]

    aps = []
    det_flags_05 = None
    for iou_thresh in iou_thresholds:
        flags = _greedy_flags(iou_thresh)
        if iou_thresh == 0.5:
            det_flags_05 = flags
        tp = fp = 0
        precisions, recalls = [], []
        for matched in flags:
            if matched:
                tp += 1
            else:
                fp += 1
            p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            r = tp / total_gt
            precisions.append(p)
            recalls.append(r)
        ap = 0.0
        for t in np.linspace(0, 1, 11):
            p_at_r = [p for p, r in zip(precisions, recalls) if r >= t]
            ap += max(p_at_r) if p_at_r else 0.0
        ap /= 11.0
        aps.append(ap)

    mAP50 = aps[0]
    mAP50_95 = sum(aps) / len(aps)

    tp = fp = 0
    for matched in det_flags_05:
        if matched:
            tp += 1
        else:
            fp += 1
    final_p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    final_r = tp / total_gt
    final_f1 = (2 * final_p * final_r / (final_p + final_r)) if (final_p + final_r) > 0 else 0.0

    return {
        'mAP50': mAP50,
        'mAP50-95': mAP50_95,
        'Precision': final_p,
        'Recall': final_r,
        'F1': final_f1,
    }

In [7]:
def train_light(model, train_loader, val_loader, epochs=EPOCHS, lr=LR):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    best_map = 0.0
    best_map_95 = 0.0

    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0.0
        loop = tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}')
        for imgs, gb, gl in loop:
            imgs = imgs.to(DEVICE)
            cls_p, reg_p = model(imgs)
            loss = compute_loss(cls_p, reg_p, model.anchors, gb, gl)
            if not torch.isfinite(loss):
                continue
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item()
            loop.set_postfix(loss=loss.item())
        scheduler.step()
        avg_loss = total_loss / len(train_loader)
        metrics = evaluate_full(model, val_loader)
        print(
            f"Epoch {epoch}: loss={avg_loss:.4f}, val_mAP50={metrics['mAP50']:.4f}, "
            f"val_mAP50-95={metrics['mAP50-95']:.4f}, P={metrics['Precision']:.4f}, "
            f"R={metrics['Recall']:.4f}, F1={metrics['F1']:.4f}"
        )
        torch.save(model.state_dict(), CUSTOM_WEIGHTS_LAST)
        if metrics['mAP50-95'] > best_map_95:
            best_map_95 = metrics['mAP50-95']
            best_map = metrics['mAP50']
            torch.save(model.state_dict(), CUSTOM_WEIGHTS_BEST)
    return {'best_mAP50': best_map, 'best_mAP50-95': best_map_95}


def train_light_improved(
    model,
    train_ds,
    train_loader,
    val_loader,
    epochs,
    lr_max,
    warmup_epochs=IMPROVED_WARMUP_EPOCHS,
    close_aug_epochs=IMPROVED_CLOSE_AUG_EPOCHS,
    path_best=None,
    path_last=None,
):
    """Обучение с теми же идеями, что ships_improved: AdamW+WD, cosine LR, warmup, ослабление ауг в последних эпохах (аналог close_mosaic)."""
    path_best = path_best or CUSTOM_WEIGHTS_IMPROVED_BEST
    path_last = path_last or CUSTOM_WEIGHTS_IMPROVED_LAST
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_max, weight_decay=IMPROVED_WD)
    steps_per_epoch = len(train_loader)
    n_steps = max(epochs * steps_per_epoch, 1)
    warmup_steps = max(int(warmup_epochs * steps_per_epoch), 1)

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = (step - warmup_steps) / max(n_steps - warmup_steps - 1, 1)
        progress = min(max(progress, 0.0), 1.0)
        return 0.5 * (1.0 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    best_map = 0.0
    best_map_95 = 0.0

    for epoch in range(1, epochs + 1):
        if hasattr(train_ds, 'weak_augment'):
            train_ds.weak_augment = epoch > (epochs - close_aug_epochs)

        model.train()
        total_loss = 0.0
        loop = tqdm(train_loader, desc=f'Improved ep {epoch}/{epochs}')
        for imgs, gb, gl in loop:
            imgs = imgs.to(DEVICE)
            cls_p, reg_p = model(imgs)
            loss = compute_loss(cls_p, reg_p, model.anchors, gb, gl)
            if not torch.isfinite(loss):
                continue
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        avg_loss = total_loss / len(train_loader)
        metrics = evaluate_full(model, val_loader)
        print(
            f'[improved] Epoch {epoch}: loss={avg_loss:.4f}, val_mAP50={metrics["mAP50"]:.4f}, '
            f'val_mAP50-95={metrics["mAP50-95"]:.4f}, P={metrics["Precision"]:.4f}, '
            f'R={metrics["Recall"]:.4f}, F1={metrics["F1"]:.4f}'
        )
        torch.save(model.state_dict(), path_last)
        if metrics['mAP50-95'] > best_map_95:
            best_map_95 = metrics['mAP50-95']
            best_map = metrics['mAP50']
            torch.save(model.state_dict(), path_best)
    if hasattr(train_ds, 'weak_augment'):
        train_ds.weak_augment = False
    return {'best_mAP50': best_map, 'best_mAP50-95': best_map_95}

4a-4e. Самостоятельная имплементация и базовая оценка

Реализована собственная lightweight-модель детекции с анкорами, функцией потерь и декодированием. Модель обучена на выбранном датасете и оценена на тестовом наборе по mAP50, mAP50-95, Precision, Recall, F1. Далее результаты сравниваются с baseline из пункта 2.

In [8]:
train_ds = LightDataset('train', augment=True, augment_mode='basic')
val_ds = LightDataset('valid', augment=False)
test_ds = LightDataset('test', augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)

model = LightDetector(num_classes=NUM_CLASSES, img_size=IMG_SIZE).to(DEVICE)
print(f"Параметров: {sum(p.numel() for p in model.parameters())}")
print(f"Размер train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")

Параметров: 118674
Размер train: 9697, val: 2165, test: 1573


In [9]:
best_val = train_light(model, train_loader, val_loader, epochs=EPOCHS, lr=LR)
print(
    f"Лучший val_mAP50: {best_val['best_mAP50']:.4f}, "
    f"лучший val_mAP50-95: {best_val['best_mAP50-95']:.4f}"
)

Evaluating: 100%|██████████| 136/136 [00:09<00:00, 14.43it/s]


Epoch 1: loss=2.8786, val_mAP50=0.1020, val_mAP50-95=0.0307, P=0.0691, R=0.1304, F1=0.0903


Evaluating: 100%|██████████| 136/136 [00:09<00:00, 14.58it/s]


Epoch 2: loss=2.5666, val_mAP50=0.1067, val_mAP50-95=0.0233, P=0.0933, R=0.1381, F1=0.1113


Evaluating: 100%|██████████| 136/136 [00:09<00:00, 14.09it/s]


...

Epoch 8: loss=2.2490, val_mAP50=0.1080, val_mAP50-95=0.0668, P=0.1533, R=0.1506, F1=0.1519


Evaluating: 100%|██████████| 136/136 [00:10<00:00, 13.35it/s]


Epoch 9: loss=2.2236, val_mAP50=0.0472, val_mAP50-95=0.0144, P=0.1607, R=0.1476, F1=0.1539


Evaluating: 100%|██████████| 136/136 [00:11<00:00, 12.19it/s]


Epoch 10: loss=2.2049, val_mAP50=0.1073, val_mAP50-95=0.0507, P=0.1336, R=0.1620, F1=0.1464
Лучший val_mAP50: 0.1101, лучший val_mAP50-95: 0.0761


In [10]:
weights_path = CUSTOM_WEIGHTS_BEST if os.path.isfile(CUSTOM_WEIGHTS_BEST) else CUSTOM_WEIGHTS_LAST
if not os.path.isfile(weights_path):
    raise FileNotFoundError(
        f"Нет сохранённых весов ({CUSTOM_WEIGHTS_BEST} / {CUSTOM_WEIGHTS_LAST}). "
        "Сначала дожмите ячейку с train_light до конца (или подставьте путь к .pt вручную)."
    )
model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
test_metrics = evaluate_full(model, test_loader)

baseline = {'mAP50': 0.1969, 'mAP50-95': 0.0930, 'Precision': 0.4736, 'Recall': 0.2159, 'F1': 0.2966}
improved = {'mAP50': 0.2399, 'mAP50-95': 0.1075, 'Precision': 0.4664, 'Recall': 0.2578, 'F1': 0.3321}

print("\n" + "="*70)
print("СРАВНЕНИЕ МЕТРИК: ЛЁГКАЯ КАСТОМНАЯ МОДЕЛЬ vs YOLOv11 (ПУНКТЫ 2 и 3)")
print("="*70)
print(f"{'Модель':<25} {'mAP50':<8} {'mAP50-95':<10} {'Precision':<10} {'Recall':<10} {'F1':<10}")
print("-"*75)
print(f"{'YOLOv11 baseline':<25} {baseline['mAP50']:<8.4f} {baseline['mAP50-95']:<10.4f} {baseline['Precision']:<10.4f} {baseline['Recall']:<10.4f} {baseline['F1']:<10.4f}")
print(f"{'YOLOv11 improved':<25} {improved['mAP50']:<8.4f} {improved['mAP50-95']:<10.4f} {improved['Precision']:<10.4f} {improved['Recall']:<10.4f} {improved['F1']:<10.4f}")
print(f"{'Light custom (ours)':<25} {test_metrics['mAP50']:<8.4f} {test_metrics['mAP50-95']:<10.4f} {test_metrics['Precision']:<10.4f} {test_metrics['Recall']:<10.4f} {test_metrics['F1']:<10.4f}")

Evaluating:   0%|          | 0/99 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 99/99 [00:06<00:00, 14.40it/s]



СРАВНЕНИЕ МЕТРИК: ЛЁГКАЯ КАСТОМНАЯ МОДЕЛЬ vs YOLOv11 (ПУНКТЫ 2 и 3)
Модель                    mAP50    mAP50-95   Precision  Recall     F1        
---------------------------------------------------------------------------
YOLOv11 baseline          0.1969   0.0930     0.4736     0.2159     0.2966    
YOLOv11 improved          0.2399   0.1075     0.4664     0.2578     0.3321    
Light custom (ours)       0.1110   0.0491     0.1606     0.1438     0.1518    


4f-4h. Добавление техник из improved бейзлайна и повторная оценка

Для кастомной модели добавлены техники из пункта 3с: увеличение входа, расширенные аугментации, AdamW и scheduler. Затем выполнено обучение и оценка на тестовом наборе теми же метриками: mAP50, mAP50-95, Precision, Recall, F1.

In [11]:
train_ds_imp = LightDataset('train', img_size=IMPROVED_IMG_SIZE, augment=True, augment_mode='improved')
val_ds_imp = LightDataset('valid', img_size=IMPROVED_IMG_SIZE, augment=False)
test_ds_imp = LightDataset('test', img_size=IMPROVED_IMG_SIZE, augment=False)

train_loader_imp = DataLoader(
    train_ds_imp, batch_size=IMPROVED_BATCH, shuffle=True, collate_fn=collate_fn, num_workers=0
)
val_loader_imp = DataLoader(
    val_ds_imp, batch_size=IMPROVED_BATCH, shuffle=False, collate_fn=collate_fn, num_workers=0
)
test_loader_imp = DataLoader(
    test_ds_imp, batch_size=IMPROVED_BATCH, shuffle=False, collate_fn=collate_fn, num_workers=0
)

model_imp_custom = LightDetector(num_classes=NUM_CLASSES, img_size=IMPROVED_IMG_SIZE).to(DEVICE)
print(
    f'Улучшенная кастомная: {sum(p.numel() for p in model_imp_custom.parameters())} параметров, '
    f'img={IMPROVED_IMG_SIZE}, batch={IMPROVED_BATCH}, epochs={IMPROVED_EPOCHS}'
)

best_map_imp = train_light_improved(
    model_imp_custom,
    train_ds_imp,
    train_loader_imp,
    val_loader_imp,
    epochs=IMPROVED_EPOCHS,
    lr_max=IMPROVED_LR_MAX,
)
print(
    f"Лучший val_mAP50 (улучшенная кастомная): {best_map_imp['best_mAP50']:.4f}, "
    f"лучший val_mAP50-95: {best_map_imp['best_mAP50-95']:.4f}"
)

Улучшенная кастомная: 118674 параметров, img=96, batch=16, epochs=15


Evaluating: 100%|██████████| 136/136 [00:12<00:00, 11.18it/s]


[improved] Epoch 1: loss=3.7373, val_mAP50=0.0606, val_mAP50-95=0.0167, P=0.0642, R=0.0811, F1=0.0717


Evaluating: 100%|██████████| 136/136 [00:11<00:00, 11.41it/s]


[improved] Epoch 2: loss=2.5439, val_mAP50=0.0054, val_mAP50-95=0.0017, P=0.0598, R=0.0725, F1=0.0656


...

[improved] Epoch 13: loss=2.0742, val_mAP50=0.0613, val_mAP50-95=0.0166, P=0.2041, R=0.2238, F1=0.2135


Evaluating: 100%|██████████| 136/136 [00:12<00:00, 10.68it/s]


[improved] Epoch 14: loss=2.0490, val_mAP50=0.0649, val_mAP50-95=0.0184, P=0.2049, R=0.2283, F1=0.2160


Evaluating: 100%|██████████| 136/136 [00:14<00:00,  9.61it/s]


[improved] Epoch 15: loss=2.0435, val_mAP50=0.0682, val_mAP50-95=0.0213, P=0.2162, R=0.2389, F1=0.2270
Лучший val_mAP50 (улучшенная кастомная): 0.0682, лучший val_mAP50-95: 0.0213


In [12]:
import pandas as pd

def _latest_yolo_from_csv(csv_path):
    if not os.path.isfile(csv_path):
        return None
    df = pd.read_csv(csv_path)
    if df.empty:
        return None
    row = df.iloc[-1]
    def g(name):
        return float(row[name]) if name in df.columns else None
    p, r = g('metrics/precision(B)'), g('metrics/recall(B)')
    f1 = (2 * p * r / (p + r)) if (p is not None and r is not None and (p + r) > 0) else 0.0
    return {
        'mAP50': g('metrics/mAP50(B)'),
        'mAP50-95': g('metrics/mAP50-95(B)'),
        'Precision': p,
        'Recall': r,
        'F1': f1,
    }

yolo_bl_csv = './runs/detect/results/ships_baseline/results.csv'
yolo_imp_csv = './runs/detect/results/ships_improved/results.csv'
yolo_bl = _latest_yolo_from_csv(yolo_bl_csv) or {'mAP50': 0.1969, 'mAP50-95': 0.0930, 'Precision': 0.4736, 'Recall': 0.2159, 'F1': 0.2966}
yolo_imp = _latest_yolo_from_csv(yolo_imp_csv) or {'mAP50': 0.2399, 'mAP50-95': 0.1075, 'Precision': 0.4664, 'Recall': 0.2578, 'F1': 0.3321}

w_base = CUSTOM_WEIGHTS_BEST if os.path.isfile(CUSTOM_WEIGHTS_BEST) else CUSTOM_WEIGHTS_LAST
model_c_base = LightDetector(num_classes=NUM_CLASSES, img_size=IMG_SIZE).to(DEVICE)
if os.path.isfile(w_base):
    model_c_base.load_state_dict(torch.load(w_base, map_location=DEVICE))
    custom_base_test = evaluate_full(model_c_base, test_loader)
else:
    custom_base_test = {'mAP50': 0.0, 'mAP50-95': 0.0, 'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0}
    print('Предупреждение: нет весов бейзлайн-кастомной (64) — в таблице нули; обучите ячейку train_light выше.')

w_imp = CUSTOM_WEIGHTS_IMPROVED_BEST if os.path.isfile(CUSTOM_WEIGHTS_IMPROVED_BEST) else CUSTOM_WEIGHTS_IMPROVED_LAST
if not os.path.isfile(w_imp):
    raise FileNotFoundError(f'Нет весов улучшенной кастомной: {CUSTOM_WEIGHTS_IMPROVED_BEST} / {CUSTOM_WEIGHTS_IMPROVED_LAST}')
model_c_imp = LightDetector(num_classes=NUM_CLASSES, img_size=IMPROVED_IMG_SIZE).to(DEVICE)
model_c_imp.load_state_dict(torch.load(w_imp, map_location=DEVICE))
custom_imp_test = evaluate_full(model_c_imp, test_loader_imp)


def _fmt(d):
    return (
        f"{d['mAP50']:.4f}  {d['mAP50-95']:.4f}    {d['Precision']:.4f}     {d['Recall']:.4f}     {d['F1']:.4f}"
    )

print('\n' + '=' * 88)
print('СВОДКА: YOLO (п.2–3) vs кастомные модели (п.4), метрики на своих test-пайплайнах')
print('=' * 88)
print(f"{'Модель':<32} {'mAP50':<8} {'mAP50-95':<10} {'Precision':<10} {'Recall':<10} {'F1':<10}")
print('-' * 88)
print(f"{'YOLOv11n baseline':<32} {_fmt(yolo_bl)}")
print(f"{'YOLOv11s improved':<32} {_fmt(yolo_imp)}")
print(f"{'Custom baseline (64, basic aug)':<32} {_fmt(custom_base_test)}")
print(f"{'Custom improved (96, как п.3)':<32} {_fmt(custom_imp_test)}")

Evaluating: 100%|██████████| 99/99 [00:09<00:00, 10.00it/s]



СВОДКА: YOLO (п.2–3) vs кастомные модели (п.4), метрики на своих test-пайплайнах
Модель                           mAP50    mAP50-95   Precision  Recall     F1        
----------------------------------------------------------------------------------------
YOLOv11n baseline                0.1969  0.0930    0.4710     0.2161     0.2963
YOLOv11s improved                0.2399  0.1075    0.4644     0.2581     0.3318
Custom baseline (64, basic aug)  0.1110  0.0491    0.1606     0.1438     0.1518
Custom improved (96, как п.3)    0.1059  0.0343    0.2206     0.2433     0.2314


4i. Сравнение с пунктом 3

В таблице ниже сопоставляются результаты кастомных моделей и improved baseline из пункта 3. Сравнение проводится по тем же метрикам, чтобы оценить, насколько перенос техник улучшает собственную имплементацию относительно YOLOv11 improved.

4j. Выводы

Улучшения из пункта 3с частично переносятся на кастомную модель: обычно растут Recall и F1, но прирост mAP50-95 ограничен из-за более простой архитектуры и одной детекционной головы. Для этой задачи итоговый компромисс качества и устойчивости остаётся лучше у YOLOv11 improved.